In [ ]:
# Run timetables.ipynb first to generate the transport stop frequency files

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
from tqdm import tqdm

from paths import paths, prepare_preprocessing
from preprocessing.src.constants import CRS_GEO, CRS_METRIC, LSOA_IDS, REGION
from preprocessing.src.geo import get_buffered_region, intersecting_features
from preprocessing.src.isochrone import extract_zones_by_duration
from preprocessing.src.threading import parallelize

In [ ]:
tqdm.pandas()

In [ ]:
prepare_preprocessing()

In [ ]:
# Estimates for transport capacity and speed
CAPACITY = {'bus': 80, 'metro': 1000, 'rail': 1000, 'tram': 200, 'boat': 220, 'light_rail': 700}
SPEED = {'bus': 15, 'rail': 40, 'metro': 33, 'tram': 22, 'boat': 25, 'light_rail': 30}

In [ ]:
# Load data and merge into a single GDF
lsoa_base = gpd.read_parquet(paths.preprocessing.shared.lsoa)

bus = pd.read_csv(paths.preprocessing.transport.bus)
rail = pd.read_csv(paths.preprocessing.transport.rail)
tfl = pd.read_csv(paths.preprocessing.transport.tfl)

bus['stop_type'] = 'bus'
rail['stop_type'] = 'rail'

conditions = [
    tfl['stop_name'].str.contains('Underground', case = False),
    tfl['stop_name'].str.contains('Pier', case = False),
    tfl['stop_name'].str.contains('Tram', case = False),
    tfl['stop_name'].str.contains('DLR', case = False)
]
choices = ['metro', 'boat', 'tram', 'light_rail']
tfl['stop_type'] = np.select(conditions, choices, default = 'tfl')
tfl = tfl[tfl['stop_type'] != 'tfl']

transport_df = pd.concat([bus, rail, tfl], ignore_index = True)
transport_gdf = gpd.GeoDataFrame(
    transport_df, geometry = gpd.points_from_xy(transport_df['stop_lon'], transport_df['stop_lat']), crs = CRS_GEO
)

In [ ]:
# Filter to stops within the study region
area = get_buffered_region(REGION, 5000, CRS_GEO)
stops_in_area = transport_gdf[transport_gdf.geometry.within(area)]
stops_in_area_metric = stops_in_area.to_crs(CRS_METRIC)

In [ ]:
# Weight transport stops based on frequency, capacity, and speed
def weight_transport_stops(df):
    df = df.copy()

    df['capacity'] = df['stop_type'].map(CAPACITY)
    df['speed'] = df['stop_type'].map(SPEED)

    min_cap = min(CAPACITY.values())
    min_spd = min(SPEED.values())
    df['cap_mult'] = df['capacity'] / min_cap
    df['speed_mult'] = df['speed'] / min_spd

    df['weight'] = np.log1p(np.sqrt(df['avg_per_day'] * df['cap_mult'] * df['speed_mult']))

    return df

stops_in_area_metric = weight_transport_stops(stops_in_area_metric)

In [ ]:
STOP_TYPES = list(CAPACITY.keys())

def process_file(lsoa_id):
    candidates = {}
    results = {}

    for minutes, zones in extract_zones_by_duration(lsoa_id, buffer = 15, crs = CRS_METRIC).items():
        scores = []
        counts_list = []
        fractions_list = []

        for seed, zone in zones:
            stops_in_zone = intersecting_features(stops_in_area_metric, zone, candidates.get(seed))
            candidates[seed] = stops_in_zone
            zone_area = zone.area / 1e6
            scores.append(stops_in_zone['weight'].sum())
            counts = stops_in_zone['stop_type'].value_counts().reindex(STOP_TYPES, fill_value = 0)
            counts_list.append(counts / zone_area)
            total = counts.sum()
            fractions_list.append(counts / total if total > 0 else pd.Series(0, index = STOP_TYPES))

        results[minutes] = {
            'lsoa_id': lsoa_id,
            'transport_score': np.mean(scores) if scores else 0,
            'transport_stop_counts': pd.DataFrame(counts_list).mean().to_dict(),
            'transport_stop_fractions': pd.DataFrame(fractions_list).mean().to_dict()
        }

    return results

results = parallelize(process_file, LSOA_IDS)

In [ ]:
def add_transport_score(gdf):
    gdf['transport_score'] = (gdf['transport_score'] - gdf['transport_score'].mean()) / gdf['transport_score'].std(
        ddof = 0
    )

    return gdf

component_data = {}

for minutes in (20, 15, 10):
    duration_results = [result[minutes] for result in results]
    transport_stats = pd.DataFrame(duration_results).set_index('lsoa_id')

    counts_df = pd.DataFrame(transport_stats['transport_stop_counts'].tolist(), index = transport_stats.index).add_prefix(
        'transport_stop_count_'
    )
    fractions_df = pd.DataFrame(
        transport_stats['transport_stop_fractions'].tolist(), index = transport_stats.index
    ).add_prefix('transport_stop_fraction_')

    transport_stats = pd.concat([transport_stats[['transport_score']], counts_df, fractions_df], axis = 1)

    lsoa = lsoa_base.merge(
        add_transport_score(transport_stats), left_on = 'LSOA21CD', right_index = True, how = 'left'
    ).fillna(0)
    component_data[minutes] = lsoa

In [ ]:
columns = ['LSOA21CD', 'geometry', 'transport_score', *counts_df.columns.tolist(), *fractions_df.columns.tolist()]

for minutes, lsoa in component_data.items():
    lsoa[columns].to_parquet(paths.preprocessing.walkability[minutes].transport, index = False)

In [ ]:
for minutes, lsoa in component_data.items():
    FILES = paths.preprocessing.walkability[minutes]
    
    m = lsoa.explore(
        column = 'transport_score',
        cmap = 'viridis',
        scheme = 'quantiles',
        k = 10,
        legend = True,
        name = 'Transport stop scores per LSOA',
        tiles = 'CartoDB positron',
        style_kwds = {'weight': 0.5, 'lineOpacity': 0.5, 'fillOpacity': 0.375},
    )

    m.save(FILES.transport_map)

m